In [1]:
import glob
import pandas as pd
import numpy as np
import torch
from collections import defaultdict

CSV_PATTERN = "/project2/ll_774_951/midterm/*/*.csv"
SOURCE_GRAPH = "/home1/eibl/gfm/prodigy/midterm/graph_co_retweet/graph_data.pt"
OUTPUT_PATH  = "/home1/eibl/gfm/prodigy/midterm/graph_co_retweet/graph_data_pseudo.pt"

COLS = ['userid', 'hashtag', 'rt_hashtag', 'text', 'location', 'description', 'urls_list', 'rt_urls_list', 'date']

In [2]:
files = sorted(glob.glob(CSV_PATTERN))
print(f"Found {len(files)} CSV files")

chunks = []
skipped = 0
for f in files:
    try:
        chunks.append(pd.read_csv(f, usecols=COLS, engine='python', on_bad_lines='skip'))
    except Exception as e:
        print(f"Skipping {f}: {e}")
        skipped += 1
df = pd.concat(chunks, ignore_index=True)
df['userid'] = pd.to_numeric(df['userid'], errors='coerce')
df = df.dropna(subset=['userid']).copy()
df['userid'] = df['userid'].astype(int)
print(f"Skipped files: {skipped}")
print(f"Total rows: {len(df):,}  |  Unique users: {df['userid'].nunique():,}")

Found 492 CSV files
Skipped files: 0
Total rows: 1,790,252  |  Unique users: 483,933


In [6]:
df.to_parquet('instagram.parquet')

In [7]:
domain_pat = '^https?://(www\.)?([\w-]+\.)*([\w-]+\.\w{1,3})'
urls = df['urls_list'].fillna('[]').apply(eval).explode().str['expanded_url'].str.extract(domain_pat)[2].groupby(level=0).agg(list)
rt_urls = df['rt_urls_list'].fillna('[]').apply(eval).explode().str['expanded_url'].str.extract(domain_pat)[2].groupby(level=0).agg(list)
# tt['qtd_urls_list'].fillna('[]').apply(eval).explode().str['expanded_url'].str.extract(domain_pat)[1].value_counts()
# tt['media_urls'].fillna('[]').apply(eval).explode().dropna()

df['urls'] = urls
df['rt_urls'] = rt_urls

df['urls'] = df.apply(lambda x: x.urls + x.rt_urls, 1).explode().dropna().groupby(level=0).agg(list)

In [8]:
df.description.str.len().max()

257.0

In [26]:
media = """@ABC
abcnews.go.com
2
@BBCWorld
bbc.com
3
@BreitbartNews
breitbart.com
5
@BostonGlobe
bostonglobe.com
2
@businessinsider
businessinsider.com
3
@BuzzFeedNews
buzzfeednews.com
1
@CBSNews
cbsnews.com
2
@chicagotribune
chicagotribune.com
3
@CNBC
cnbc.com
3
@ CNN
cnn.com
2
@DailyCaller
dailycaller.com
5
@DailyMail
dailymail.co.uk
5
@FoxNews
foxnews.com
4
@HuffPost
huffpost.com
1
InfoWars*
infowars.com
5
@latimes
latimes.com
2
@MSNBC
msnbc.om
1
@NBCNews
nbcnews.com
2
@nytimes
nytimes.com
2
@NPR
npr.org
3
@OANN
oann.com
4
@PBS
pbs.org
3
@Reuters
reuters.com
3
@guardian
theguardian.com
2
@USATODAY
usatoday.com
3
@YahooNews
yahoo.com
2
@VICE
vice.com
1
@washingtonpost
washingtonpost.com
2
@WSJ
wsj.com
3"""
outlets = media.split('\n')[1::3]
ratings = media.split('\n')[2::3]
ratings = pd.Series(dict(zip(outlets, ratings)))
index = ratings.index.str.split('.').str[0]
ratings.index = index

ratings = ratings.astype(float)
dem_media = ratings[ratings.lt(3)].index
dem_media = '|'.join(dem_media)
rep_media = ratings[ratings.gt(3)].index
rep_media = '|'.join(rep_media)

df['dem_url_count'] = df.urls.astype(str).str.count(dem_media)
df['rep_url_count'] = df.urls.astype(str).str.count(rep_media)

In [27]:
df['date'] = pd.to_datetime(df['date'], format='%a %b %d %H:%M:%S %z %Y')
udf = df.sort_values('date', ascending=False).drop_duplicates('userid', keep='first')
udf = udf.set_index('userid')

In [28]:
udf['dem_url_count'] = df.groupby('userid')['dem_url_count'].sum()
udf['rep_url_count'] = df.groupby('userid')['rep_url_count'].sum()

In [29]:
red = [
"maga",
"voteredtosaveamerica",
"votered",
"redwavecoming",
'democratsAreTheProblem'
]

blue = [
"voteblue",
"voteblue2022",
"votebluetosavedemocracy",
"votebluetosaveamerica",
"votebluein2022",
"votebluenomatterwho",
"votebluefordemocracy",
"votebluetoprotectwomen",
"voteblueforwomensrights",
"votebluetoprotectyourrights",
"voteblueforsomanyreasons",
"votebluetoendtheinsanity",
"votebluenotq",
"votebluedownballot",
"votebluedownballotlocalstatefederal",
"votebluetosavesocialsecurity",
"votebluetosavesocialsecurityandmedicare",
"votebluetosaveourkids",
"bluewave",
"bluewave2022",
"bluecrew",
"bluevoters",
"ourbluevoice",
"bluein22",
"proudblue22",
"demvoice1",
"wtpblue",
"democratsdeliver",
"demsact",
"voteouteveryrepublican",
"stopvotingforrepublicans",
"neverrepublicanagain",
"republicansaretheproblem",
"republicanwaronwomen",
"goptraitorstodemocracy",
"gopliesabouteverything",
'magaidiots'
]

red_pat = '\#' + "|\#".join(red)
udf['rep_hash_count'] = udf.description.str.lower().str.count(red_pat)
blue_pat = '\#' + "|\#".join(blue)
udf['dem_hash_count'] = udf.description.str.lower().str.count(blue_pat)

In [61]:
udf['label'] = -1
i = udf[['rep_hash_count', 'rep_url_count']].sum(1).gt(0)
udf.loc[i,'label'] = 0
i = udf[['dem_hash_count', 'dem_url_count']].sum(1).gt(0)
udf.loc[i,'label'] = 1

uid_to_label = udf['label'].replace({-1: np.nan}).dropna().to_dict()

label_names = {0: 'rep', 1: 'dem'}

In [62]:
raw = torch.load(SOURCE_GRAPH, map_location='cpu')
user_ids = raw['user_ids']
num_nodes = len(user_ids)
print(f"Graph nodes: {num_nodes:,}")

# Build node-level label tensor (-1 = unlabeled)
y_pseudo = torch.full((num_nodes,), -1, dtype=torch.long)
labeled = 0
for node_idx, uid in enumerate(user_ids):
    try:
        uid_int = int(uid)
    except (ValueError, TypeError):
        continue
    if uid_int in uid_to_label:
        y_pseudo[node_idx] = uid_to_label[uid_int]
        labeled += 1

print(f"Labeled nodes: {labeled:,} / {num_nodes:,} ({100*labeled/num_nodes:.1f}%)")

# Save as a new graph_data.pt with pseudo labels swapped in for y
out = dict(raw)
out['y'] = y_pseudo
out['label_names'] = label_names
torch.save(out, OUTPUT_PATH)
print(f"Saved to {OUTPUT_PATH}")

Graph nodes: 547,504
Labeled nodes: 38,489 / 547,504 (7.0%)
Saved to /home1/eibl/gfm/prodigy/midterm/graph_co_retweet/graph_data_pseudo.pt


In [59]:
 print(y_pseudo[y_pseudo >= 0].bincount())

tensor([ 8374, 30115])


In [60]:
label_names

{-1: 'none', 0: 'rep', 1: 'dem'}